In [16]:
import json, re
from pathlib import Path
from datetime import datetime, date
from collections import defaultdict

import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data" / "raw"
DATA_CONFIG = ROOT / "data" / "monitoring_config" / "test_id_to_target_host_mapper.json"

GRAPH_SAVE_FOLDER = './results'
GRAPH_RESULTS_DIR = Path(GRAPH_SAVE_FOLDER)


# EWMA control chart parameters
ALPHA = 0.075      # smoothing factor (alpha in (0, 1])
SIGMA_MULT = 3     # control limit multiplier (UCL/LCL = ewma +/- SIGMA_MULT * sigma)



def get_resource_time_ms(r):
    try:
        return float(r.get("time", 0))
    except (TypeError, ValueError):
        return 0.0


print('Configuration OK.')


Configuration OK.


In [17]:
# LOADING CONFIG AND DISCOVERY OF RAW DATASET 

with open(DATA_CONFIG, encoding='utf-8') as f:
    data_config = json.load(f)

# build  mappings test_id -> target_host
testid_to_host = {e['test_id']: e['target_host'] for e in data_config}
host_to_testid = {e['target_host']: e['test_id'] for e in data_config}

# discover all available JSON data files and their dates from data/raw directory
json_files = sorted(DATA_DIR.glob("webapp.http_dynamic.single_page.tranco.*.json"))


def extract_date(p: Path) -> str | None:
    # extract date string (YYYY-MM-DD) from a file name."""
    m = re.search(r'(\d{4}-\d{2}-\d{2})', p.stem)
    return m.group(1) if m else None


available_dates = sorted({d for p in json_files if (d := extract_date(p))})

print(f"Available date range: {available_dates[0]} → {available_dates[-1]}")
print(f"Sites loaded from config: {len(data_config)}")

Available date range: 2026-02-24 → 2026-02-28
Sites loaded from config: 62


In [18]:

def compute_content_time(load_time, waterfall_analysis):
    """Derive 'content_time' from a single snapshot's waterfall

      1. t0 = the lowest start_time among all resources in the waterfall.
      2. For every resource r: rel_end_r = (start_time_r - t0) + time_r
      3. Keep only resources with rel_end_r <= load_time 
      4. content_time = max(rel_end_r) - min(start_time_r - t0) over the
         filtered set.

    """
    if waterfall_analysis is None or load_time is None or pd.isna(load_time):
        return np.nan
    if not isinstance(waterfall_analysis, (list, tuple)) or len(waterfall_analysis) == 0:
        return np.nan

    starts = []
    for r in waterfall_analysis:
        try:
            starts.append(float(r["start_time"]))
        except (KeyError, TypeError, ValueError):
            continue
    if not starts:
        return np.nan
    t0 = min(starts)

    rel_starts, rel_ends = [], []
    for r in waterfall_analysis:
        try:
            start = float(r["start_time"])
            dur = get_resource_time_ms(r)
        except (KeyError, TypeError, ValueError):
            continue
        rel_start = start - t0
        rel_end = rel_start + dur
        if rel_end <= load_time:
            rel_starts.append(rel_start)
            rel_ends.append(rel_end)

    if not rel_ends:
        return np.nan
    return max(rel_ends) - min(rel_starts)


def add_content_time_column(g):
    """Adds a 'content_time' column to a
    per-test_id dataframe that has 'load_time' and 'waterfall_analysis'."""
    g = g.copy()
    g["content_time"] = [
        compute_content_time(lt, wf)
        for lt, wf in zip(g["load_time"], g["waterfall_analysis"])
    ]
    return g


class EWMAControlChart:
    #Incremental EWMA control chart.
    def __init__(self, alpha=ALPHA):
        self.alpha = alpha
        self._ewma = None
        self._ex2 = None

    def push(self, x):
        if pd.isna(x):
            return  
        if self._ewma is None:
            self._ewma = x
            self._ex2 = x * x
        else:
            a = self.alpha
            self._ewma = a * x + (1 - a) * self._ewma
            self._ex2 = a * (x * x) + (1 - a) * self._ex2

    def mean(self):
        return self._ewma

    def stddev(self):
        if self._ewma is None:
            return None
        var = max(self._ex2 - self._ewma ** 2, 0.0) 
        return var ** 0.5


def run_ewma(series):
    """Runs the EWMA control chart over a (already time-interpolated) series
    and returns ewma/upper/lower/anomaly-over/anomaly-under series aligned to
    the same index."""
    chart = EWMAControlChart(ALPHA)

    ewma_vals, upper_vals, lower_vals, over_flags, under_flags = [], [], [], [], []
    for x in series:
        chart.push(x)
        mean = chart.mean()
        std = chart.stddev()
        if mean is None:
            ewma_vals.append(np.nan)
            upper_vals.append(np.nan)
            lower_vals.append(np.nan)
            over_flags.append(False)
            under_flags.append(False)
            continue
        ucl = mean + SIGMA_MULT * std
        lcl = mean - SIGMA_MULT * std
        ewma_vals.append(mean)
        upper_vals.append(ucl)
        lower_vals.append(lcl)
        over_flags.append(bool(x > ucl) if not pd.isna(x) else False)
        under_flags.append(bool(x < lcl) if not pd.isna(x) else False)

    idx = series.index
    return (
        pd.Series(ewma_vals, index=idx),
        pd.Series(upper_vals, index=idx),
        pd.Series(lower_vals, index=idx),
        pd.Series(over_flags, index=idx),
        pd.Series(under_flags, index=idx),
    )


def diagnose_network_vs_rendering(load_over, content_over):
    """Decision tree, applied to the
    EWMA over-threshold flags of load_time and Content Time:

      - Network Problem: anomaly on BOTH load_time and Content Time.
      - Slow Rendering:   anomaly on load_time ONLY (Content Time normal).

    Both inputs are boolean series aligned on the same (interpolated) index.
    Returns two DatetimeIndex-like indexes: network_idx, slow_idx.
    """
    content_aligned = content_over.reindex(load_over.index, fill_value=False)
    network_idx = load_over[load_over & content_aligned].index
    slow_idx = load_over[load_over & ~content_aligned].index
    return network_idx, slow_idx



def _finish_figure(fig, out_path, save, show):
    fig.tight_layout()
    if save and out_path is not None:
        fig.savefig(out_path, dpi=150)
    if show:
        plt.show()
    else:
        plt.close(fig)



def diagnose_and_plot(g, target,test_id, show_content_time_chart=False,
                       save=True, show=True, out_dir=None):
    """Main entry point for Cell 4's Run button.

    g: per-test_id dataframe, indexed by timestamp, with columns
       'load_time' and 'waterfall_analysis'.
    test_id: label used in the plot title and output filename.
    show_content_time_chart: if True, also renders a standalone EWMA chart
       for content_time on its own figure.
    save / show: mirror the save_graphs / show_graphs checkboxes — save
       writes a PNG to out_dir, show calls plt.show(); either can be off.
    out_dir: directory to save PNGs into (defaults to GRAPH_RESULTS_DIR).


    Returns (g, out_path, anomaly_counts) so callers (e.g. Cell 4) can print
    a short summary, matching the shape the old plot_page() used to give.
    """
    out_dir = Path(out_dir) if out_dir is not None else GRAPH_RESULTS_DIR
    if save:
        out_dir.mkdir(parents=True, exist_ok=True)

    g = add_content_time_column(g)

    s_load = g["load_time"].interpolate(method="time")
    s_content = g["content_time"].interpolate(method="time")

    load_ewma, load_ucl, load_lcl, load_over, load_under = run_ewma(s_load)
    content_ewma, content_ucl, content_lcl, content_over, content_under = run_ewma(s_content)

    network_idx, slow_idx = diagnose_network_vs_rendering(load_over, content_over)

    # ==== Main figure: load_time + EWMA band + diagnosis markers ====
    fig = plt.figure(figsize=(12, 5))
    plt.plot(s_load.index, s_load, label="load_time", marker='o', linestyle='', zorder=1, alpha=0.8)
    plt.plot(s_load.index, load_ucl, label=f"Upper (+{SIGMA_MULT}\u03c3)", color='green', linestyle='--')
    plt.plot(s_load.index, load_lcl, label=f"Lower (-{SIGMA_MULT}\u03c3)", color='green', linestyle='--')
    plt.scatter(network_idx, s_load.loc[network_idx], color='red', zorder=2, label="Network problem")
    plt.scatter(slow_idx, s_load.loc[slow_idx], color='orange', zorder=2, label="Slow rendering")
    plt.title(f"{target}  |  TestId: {test_id}  |  EWMA diagnosis for load_time")
    plt.xlabel("Time")
    plt.xticks(rotation=45, ha="right", fontsize=10)
    plt.ylabel("load_time (ms)")
    plt.legend()
    plt.grid(True)

    host_slug = (target
                     .replace('https://', '').replace('http://', '')
                     .replace('.', '_').replace('/', '_'))
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    out_path = out_dir / f'{ts}.{host_slug}.{test_id}_load_time_diagnosis.png'
    _finish_figure(fig, out_path, save, show)

    # Optional standalone Content Time EWMA chart
    if show_content_time_chart:
        fig2 = plt.figure(figsize=(12, 4))
        plt.plot(s_content.index, s_content, label="Content Time", marker='o', linestyle='', zorder=1, alpha=0.8)
        plt.plot(s_content.index, content_ucl, label=f"Upper (+{SIGMA_MULT}\u03c3)", color='green', linestyle='--')
        plt.plot(s_content.index, content_lcl, label=f"Lower (-{SIGMA_MULT}\u03c3)", color='green', linestyle='--')
        plt.scatter(
            content_over[content_over].index,
            s_content[content_over],
            color='purple', zorder=2, label=f"Anomaly >{SIGMA_MULT}\u03c3",
        )
        plt.title(f"{target}  |  TestId: {test_id}  | EWMA for Content Time")
        plt.xlabel("Time")
        plt.xticks(rotation=45, ha="right", fontsize=10)
        plt.ylabel("Content Time (ms)")
        plt.legend()
        plt.grid(True)
        
        host_slug2 = (target
                     .replace('https://', '').replace('http://', '')
                     .replace('.', '_').replace('/', '_'))
        ts2 = datetime.now().strftime('%Y%m%d_%H%M%S')
        out_path2 = out_dir / f'{ts2}.{host_slug2}.{test_id}_content_time_ewma.png'
        _finish_figure(fig2, out_path2, save, show)

    anomaly_counts = {
        "network": len(network_idx),
        "slow_rendering": len(slow_idx),
    }
    return g, out_path, anomaly_counts

print(" Content Time, EWMA Control Chart, Diagnosis and Plotting loaded.")


 Content Time, EWMA Control Chart, Diagnosis and Plotting loaded.


In [19]:

date_objects = [date.fromisoformat(d) for d in available_dates]
options = [(d.strftime(' %Y-%m-%d '), d) for d in date_objects]

date_range = widgets.SelectionRangeSlider(
    options=options,
    index=(0, len(options) - 1),
    description='Date range:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='900px')
)

host_labels = [e['target_host'] for e in data_config]

site_select = widgets.SelectMultiple(
    options=host_labels,
    value=host_labels,
    rows=min(len(host_labels), 12),
    description='Sites:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='500px')
)

btn_all_sites = widgets.Button(description='Select all', button_style='')
btn_none_sites = widgets.Button(description='Clear selection', button_style='')

content_time_chart = widgets.Checkbox(
    value=False,
    description='Show standalone Content Time EWMA chart',
    indent=False
)

save_graphs = widgets.Checkbox(
    value=True,
    description='Save graphs to ' + GRAPH_SAVE_FOLDER,
    indent=False
)

show_graphs = widgets.Checkbox(
    value=True,
    description='Show graphs in console',
    indent=False
)

run_btn = widgets.Button(
    description='▶ Run analysis',
    button_style='success',
    layout=widgets.Layout(width='200px', height='40px')
)

output = widgets.Output()


# BUTTON CALLBACKS

def on_all_sites(b):
    site_select.value = host_labels


def on_none_sites(b):
    site_select.value = []


def on_run(b):
    date_from, date_to = date_range.value
    selected_hosts = list(site_select.value)
    selected_test_ids = {host_to_testid[h] for h in selected_hosts if h in host_to_testid}

    selected_dates = [
        d for d in available_dates
        if date.fromisoformat(d) >= date_from and date.fromisoformat(d) <= date_to
    ]
    selected_files = [
        p for p in json_files
        if (d := extract_date(p)) and d in selected_dates
    ]

    with output:
        output.clear_output(wait=True)

        print(f"Selected date range : {selected_dates[0]} – {selected_dates[-1]}")
        print(f"Selected sites      : {len(selected_test_ids)}")

        # LOAD
        pages_raw = defaultdict(list)

        for filepath in selected_files:
            with open(filepath, encoding='utf-8') as f:
                print(f"Loading file to analyze ...{filepath}")
                for raw in f:
                    raw = raw.strip()
                    if not raw:
                        continue
                    try:
                        obj = json.loads(raw)
                    except json.JSONDecodeError:
                        continue

                    result = obj.get('Result')
                    if not result or result.get('status') != 'completed':
                        continue

                    pages = result.get('visited_pages', [])
                    if not pages:
                        continue

                    test_id = obj.get('Meta', {}).get('TestId', '')
                    if test_id not in selected_test_ids:
                        continue

                    host = testid_to_host.get(test_id, test_id)
                    timestamp = (obj.get('Meta', {}).get('Timestamp')
                                 or obj.get('Meta', {}).get('timestamp')
                                 or filepath.stem)

                    page = pages[0]
                    
                    # only start_time and time are needed for compute_content_time;
            
                    slim_waterfall = [
                        {'start_time': r.get('start_time'), 'time': r.get('time')}
                        for r in page.get('waterfall_analysis', [])
                    ]


                    pages_raw[host].append({
                        'timestamp': pd.to_datetime(str(timestamp)),
                        'test_id': test_id,
                        'load_time': page.get('load_time'),
                        'waterfall_analysis': slim_waterfall,
                    })

        print(f"\nLoaded sites: {len(pages_raw)}\n")
        for host, snaps in pages_raw.items():
            print(f"  {host:45s}  {len(snaps)} snapshots")

        # ANALYSE AND PLOT
        GRAPH_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

        out_path = None
        for target, snapshots in sorted(pages_raw.items()):
            test_id = host_to_testid.get(target, 'unknown')
            print(f'\n[{target}]  TestId: {test_id}  ({len(snapshots)} snapshots)')

            g = pd.DataFrame(snapshots).sort_values('timestamp').set_index('timestamp')
            g = g[['load_time', 'waterfall_analysis']]

            g_out, out_path, counts = diagnose_and_plot(
                g, target,test_id,
                content_time_chart.value,
                save_graphs.value,
                show_graphs.value,
            )

            total_anom = counts.get('network', 0) + counts.get('slow_rendering', 0)
            counts_str = f"network: {counts.get('network', 0)}  slow_rendering: {counts.get('slow_rendering', 0)}"
            print(f'[INFO]  anomalies: {total_anom}   {counts_str}')
            if out_path:
                print(f'[INFO]  saved: {out_path.name}')

        print("Done.")
        if out_path:
            print(f'\nCharts saved to: {GRAPH_RESULTS_DIR.resolve()}')


# DISPLAY AND REGISTER CALLBACKS

btn_all_sites.on_click(on_all_sites)
btn_none_sites.on_click(on_none_sites)
run_btn.on_click(on_run)

display(
    widgets.Label('── Select Date Range ──'),
    date_range,
    widgets.Label('── Select Websites ──'),
    widgets.Label('Multiple values can be selected with Shift and/or Ctrl (or Cmd) + click.'),
    site_select,
    widgets.HBox([btn_all_sites, btn_none_sites]),
    content_time_chart,
    show_graphs,
    save_graphs,
    widgets.Label(''),
    run_btn,
    output
)


Label(value='── Select Date Range ──')

SelectionRangeSlider(description='Date range:', index=(0, 4), layout=Layout(width='900px'), options=((' 2026-0…

Label(value='── Select Websites ──')

Label(value='Multiple values can be selected with Shift and/or Ctrl (or Cmd) + click.')

SelectMultiple(description='Sites:', index=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, …

...

Checkbox(value=True, description='Show graphs in console', indent=False)

Checkbox(value=True, description='Save graphs to ./results', indent=False)

Label(value='')

Button(button_style='success', description='▶ Run analysis', layout=Layout(height='40px', width='200px'), styl…

Output()